<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np

# 1. 구글 드라이브 연결 및 2025년 청소년건강행태조사 원시자료 로드
drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')

# 2. 변수 생성 (행을 삭제하지 않고 전체 54,170명 유지!)
# 고등학생 도메인 (GRADE 4, 5, 6 = 고1, 고2, 고3)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 아침식사 결식 변수 (F_BR: 1~3은 주 5회 이상 결식 -> 1, 4~8은 비결식 -> 0)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notna(), 0, np.nan))

# BMI 및 비만 변수 (BMI >= 25 -> 1, < 25 -> 0)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notna(), 0, np.nan))

# 최종 유효 분석 대상자 표시 (고등학생 & 결식여부 유효 & BMI 유효)
df['in_analysis'] = ((df['domain_hs'] == 1) & df['bf_skip'].notna() & df['bmi'].notna()).astype(int)

# B-1 흐름도 인원 확인
print("=== [B-1] 분석 대상자 선정 흐름도 확인 ===")
print(f"1단계 (전체 표본): {len(df):,}명")
print(f"2단계 (고등학생 도메인): {df['domain_hs'].sum():,}명 (제외 중학생: {(df['domain_hs'] == 0).sum():,}명)")
print(f"3단계 (고등학생 중 HT/WT 결측 제외): {((df['domain_hs'] == 1) & df['bmi'].isna()).sum():,}명")
print(f"4단계 (최종 유효 분석 대상자): {df['in_analysis'].sum():,}명")
print(f"  - 결식군: {((df['in_analysis'] == 1) & (df['bf_skip'] == 1)).sum():,}명")
print(f"  - 비결식군: {((df['in_analysis'] == 1) & (df['bf_skip'] == 0)).sum():,}명")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== [B-1] 분석 대상자 선정 흐름도 확인 ===
1단계 (전체 표본): 54,170명
2단계 (고등학생 도메인): 25,843명 (제외 중학생: 28,327명)
3단계 (고등학생 중 HT/WT 결측 제외): 766명
4단계 (최종 유효 분석 대상자): 25,077명
  - 결식군: 11,376명
  - 비결식군: 13,701명


/tmp/ipykernel_2168/4165922943.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_2168/4165922943.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_2168/4165922943.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instea

In [2]:
# 교수님 제공 설계 기반 SE 계산 함수
W, H, C = 'W', 'STRATA', 'CLUSTER'

def svy_mean(y, domain):
    """행을 지우지 않고 domain(0/1) 안의 가중 평균(비율)과 설계 기반 SE를 계산한다."""
    d = (domain == 1) & y.notna()
    w = df[W] * d
    yv = y.fillna(0)
    est = (w * yv).sum() / w.sum()
    z = w * (yv - est) / w.sum()
    t = pd.DataFrame({'h': df[H], 'c': df[C], 'z': z}).groupby(['h', 'c'])['z'].sum().reset_index()
    var = sum(len(g) / (len(g) - 1) * ((g['z'] - g['z'].mean()) ** 2).sum()
              for _, g in t.groupby('h') if len(g) > 1)
    return est, var ** 0.5

# 도메인 정의 (전체 / 결식군 / 비결식군)
dom_all = (df['in_analysis'] == 1)
dom_skip = dom_all & (df['bf_skip'] == 1)
dom_nonskip = dom_all & (df['bf_skip'] == 0)

# 범주형 행(인원, 가중%, SE) 포맷팅 함수
def format_cat_row(label, cond):
    y = cond.astype(float)
    n_all = int((cond & dom_all).sum())
    p_all, se_all = svy_mean(y, dom_all.astype(int))

    n_s = int((cond & dom_skip).sum())
    p_s, se_s = svy_mean(y, dom_skip.astype(int))

    n_ns = int((cond & dom_nonskip).sum())
    p_ns, se_ns = svy_mean(y, dom_nonskip.astype(int))

    return {
        '특성': label,
        '전체': f"{n_all:,} ({p_all*100:.2f}% [SE {se_all*100:.2f}])",
        '결식군': f"{n_s:,} ({p_s*100:.2f}% [SE {se_s*100:.2f}])",
        '비결식군': f"{n_ns:,} ({p_ns*100:.2f}% [SE {se_ns*100:.2f}])"
    }

# 연속형 행(가중평균, SE) 포맷팅 함수
def format_cont_row(label, var_series):
    m_all, se_all = svy_mean(var_series, dom_all.astype(int))
    m_s, se_s = svy_mean(var_series, dom_skip.astype(int))
    m_ns, se_ns = svy_mean(var_series, dom_nonskip.astype(int))

    return {
        '특성': label,
        '전체': f"{m_all:.2f} (SE {se_all:.4f})",
        '결식군': f"{m_s:.2f} (SE {se_s:.4f})",
        '비결식군': f"{m_ns:.2f} (SE {se_ns:.4f})"
    }

# 전체(Total) 행 별도 계산
p_skip_tot, se_skip_tot = svy_mean((df['bf_skip'] == 1).astype(float), dom_all.astype(int))
p_nonskip_tot, se_nonskip_tot = svy_mean((df['bf_skip'] == 0).astype(float), dom_all.astype(int))

rows = [
    {
        '특성': '전체 (Total)',
        '전체': f"{int(dom_all.sum()):,} (100.0%)",
        '결식군': f"{int(dom_skip.sum()):,} ({p_skip_tot*100:.2f}% [SE {se_skip_tot*100:.2f}])",
        '비결식군': f"{int(dom_nonskip.sum()):,} ({p_nonskip_tot*100:.2f}% [SE {se_nonskip_tot*100:.2f}])"
    },
    format_cat_row('남학생 (SEX==1)', df['SEX'] == 1),
    format_cat_row('여학생 (SEX==2)', df['SEX'] == 2),
    format_cat_row('고1 (GRADE==4)', df['GRADE'] == 4),
    format_cat_row('고2 (GRADE==5)', df['GRADE'] == 5),
    format_cat_row('고3 (GRADE==6)', df['GRADE'] == 6),
    format_cat_row('비비만 (BMI < 25)', df['obesity'] == 0),
    format_cat_row('비만 (BMI >= 25)', df['obesity'] == 1),
    format_cont_row('체질량지수 (BMI)', df['bmi'])
]

table1_df = pd.DataFrame(rows)
print("=== [B-2] Table 1 설계 기반 복합표본 추정 결과 ===")
display(table1_df)

print("\n[Issue 댓글 첨부용 - 남학생 행]")
print(table1_df[table1_df['특성'] == '남학생 (SEX==1)'].to_string(index=False))

=== [B-2] Table 1 설계 기반 복합표본 추정 결과 ===


,특성,전체,결식군,비결식군
0,전체 (Total),"25,077 (100.0%)","11,376 (45.68% [SE 0.49])","13,701 (54.32% [SE 0.49])"
1,남학생 (SEX==1),"12,949 (51.55% [SE 1.93])","5,697 (50.02% [SE 1.95])","7,252 (52.84% [SE 1.99])"
2,여학생 (SEX==2),"12,128 (48.45% [SE 1.93])","5,679 (49.98% [SE 1.95])","6,449 (47.16% [SE 1.99])"
3,고1 (GRADE==4),"8,526 (32.83% [SE 0.21])","3,676 (31.23% [SE 0.40])","4,850 (34.18% [SE 0.36])"
4,고2 (GRADE==5),"8,363 (33.17% [SE 0.19])","3,903 (34.13% [SE 0.42])","4,460 (32.36% [SE 0.36])"
5,고3 (GRADE==6),"8,188 (34.00% [SE 0.22])","3,797 (34.64% [SE 0.45])","4,391 (33.46% [SE 0.40])"
6,비비만 (BMI < 25),"19,933 (79.94% [SE 0.46])","8,976 (79.35% [SE 0.55])","10,957 (80.44% [SE 0.53])"
7,비만 (BMI >= 25),"5,144 (20.06% [SE 0.46])","2,400 (20.65% [SE 0.55])","2,744 (19.56% [SE 0.53])"
8,체질량지수 (BMI),22.18 (SE 0.0521),22.27 (SE 0.0569),22.11 (SE 0.0570)



[Issue 댓글 첨부용 - 남학생 행]
          특성                        전체                      결식군                     비결식군
남학생 (SEX==1) 12,949 (51.55% [SE 1.93]) 5,697 (50.02% [SE 1.95]) 7,252 (52.84% [SE 1.99])


In [3]:
print("=== [B-3] 비가중 대 가중 추정치 전체 비교 (N = 25,077) ===")

def compare_cat(label, cond):
    sub = df[dom_all]
    c_sub = cond[dom_all]
    n = int(c_sub.sum())
    unw_pct = (n / len(sub)) * 100
    w_pct, w_se = svy_mean(cond.astype(float), dom_all.astype(int))
    w_pct *= 100
    w_se *= 100
    return {
        '변수': label,
        '비가중': f"{n:,} ({unw_pct:.2f}%)",
        '가중': f"{w_pct:.2f}% (SE {w_se:.2f})",
        '차이(가중-비가중)': f"{w_pct - unw_pct:+.2f}%p"
    }

b3_rows = [
    compare_cat('아침식사 결식군', df['bf_skip'] == 1),
    compare_cat('남학생', df['SEX'] == 1),
    compare_cat('여학생', df['SEX'] == 2),
    compare_cat('고1', df['GRADE'] == 4),
    compare_cat('고2', df['GRADE'] == 5),
    compare_cat('고3', df['GRADE'] == 6),
    compare_cat('비비만 (BMI < 25)', df['obesity'] == 0),
    compare_cat('비만 (BMI >= 25)', df['obesity'] == 1),
]

bmi_sub = df.loc[dom_all, 'bmi']
unw_bmi_m = bmi_sub.mean()
unw_bmi_se = bmi_sub.std(ddof=1) / np.sqrt(len(bmi_sub))
w_bmi_m, w_bmi_se = svy_mean(df['bmi'], dom_all.astype(int))

b3_rows.append({
    '변수': 'BMI 평균',
    '비가중': f"{unw_bmi_m:.2f} (SE {unw_bmi_se:.4f})",
    '가중': f"{w_bmi_m:.2f} (SE {w_bmi_se:.4f})",
    '차이(가중-비가중)': f"{w_bmi_m - unw_bmi_m:+.2f}"
})

display(pd.DataFrame(b3_rows))

=== [B-3] 비가중 대 가중 추정치 전체 비교 (N = 25,077) ===


,변수,비가중,가중,차이(가중-비가중)
0,아침식사 결식군,"11,376 (45.36%)",45.68% (SE 0.49),+0.32%p
1,남학생,"12,949 (51.64%)",51.55% (SE 1.93),-0.09%p
2,여학생,"12,128 (48.36%)",48.45% (SE 1.93),+0.09%p
3,고1,"8,526 (34.00%)",32.83% (SE 0.21),-1.17%p
4,고2,"8,363 (33.35%)",33.17% (SE 0.19),-0.18%p
5,고3,"8,188 (32.65%)",34.00% (SE 0.22),+1.35%p
6,비비만 (BMI < 25),"19,933 (79.49%)",79.94% (SE 0.46),+0.46%p
7,비만 (BMI >= 25),"5,144 (20.51%)",20.06% (SE 0.46),-0.46%p
8,BMI 평균,22.23 (SE 0.0240),22.18 (SE 0.0521),-0.04
